# Optibound : Analysis of Modes of Convergence by Gradient Descent

In [1]:
import math
import numpy as np
import torch 
import matplotlib.pyplot as plt

#### The objective of this notebook is to demonstrate experimentally the crucial importance of the **Lipschitz Constant** in the choice of the learning rate for the gradient descent algorithm 

## 1. Gradient Descent
To minimize a loss function $f(x)$, the gradient descent iterates according to the rule of the following update : 
$$ x_{k+1} = x_k - \eta \nabla f(x_k) $$
where $\eta$ is the learning rate and $\nabla f(x_k)$ is the gradient vector evaluated at the current position.

The entire challenge of optimization lies in the choice of $\eta$: too small, and the algorithm is inefficient; too large, and it explodes.

## 2. The Lipschitz Constant ($L$)

To guarantee convergence, we rely on the concept of an $L$-smooth function. A function is called $L$-smooth if its gradient does not change too abruptly, meaning:


$$\Vert{}\nabla f(x) - \nabla f(y)\Vert{} \le L \Vert{}x - y\Vert{}$$


Physically, $L$ represents the **maximum curvature** of the function (or the largest eigenvalue of the Hessian matrix). It indicates the absolute speed limit that must not be exceeded during the descent.

## 3. The Fundamental Theorem (The 3 Regimes)

Convex optimization theory proves that the trajectory's behavior depends entirely on the ratio between $\eta$ and $L$. There are exactly three regimes:

1. **The monotonic convergence regime ($0 < \eta \le 1/L$):** The step size is perfectly adapted. The loss $f(x)$ strictly decreases at each iteration. The trajectory slides toward the minimum without ever bouncing.
2. **The oscillatory convergence regime ($1/L < \eta < 2/L$):** The step size is too large; the algorithm overcorrects and overshoots the bottom of the valley at each step. However, the error is absorbed, and the point eventually converges in a "zig-zag" pattern.
3. **The divergence regime (Explosion) ($\eta \ge 2/L$):** The mathematical limit is crossed. The overcorrection amplifies exponentially with each iteration. The trajectory is ejected from the valley, and the loss tends toward infinity.

## Experiment Objective

We will verify these three regimes on our test function `EllipticBowl` defined by $f(x, y) = 0.5 x^2 + 5 y^2$. For this analytical function, the maximum curvature is determined by the largest coefficient: **$L = 10.0$**. The theoretical limits for $\eta$ are therefore:

* Monotonic limit: $\eta = 0.10$
* Explosion limit: $\eta = 0.20$

In [2]:
from optibound.functions import EllipticBowl
from optibound.estimator import MonteCarloLipschitz
from optibound.optimizer import GradientDescent

In [3]:
cuv = EllipticBowl()
est = MonteCarloLipschitz(cuv, (-10, 10), 10000)
res = est.estimate(batch_size=2000, epsilon=1e-7)

In [4]:
res

{'L_sup': 9.999999046325684,
 'L_hat': 15.839703573125457,
 'mean': 6.4788775390625,
 'std': 2.9198522633998865,
 'valid_samples': 10000,
 'suggested_lr': 0.06313249458131634}

In [5]:
max_thresold = 10e5
tolerance = 10e-5
max_iter = 1000

lr_found = res['suggested_lr']
grad = GradientDescent(cuv, lr_found, max_iter, tolerance, max_thresold)
opt = grad.minimize(torch.tensor([8.0, 8.0]))

In [20]:
x_y = torch.stack(grad.history['x'])

In [26]:
x_y

tensor([[8.0000e+00, 8.0000e+00],
        [7.4949e+00, 2.9494e+00],
        [7.0218e+00, 1.0874e+00],
        [6.5785e+00, 4.0089e-01],
        [6.1631e+00, 1.4780e-01],
        [5.7741e+00, 5.4489e-02],
        [5.4095e+00, 2.0089e-02],
        [5.0680e+00, 7.4062e-03],
        [4.7481e+00, 2.7305e-03],
        [4.4483e+00, 1.0067e-03],
        [4.1675e+00, 3.7113e-04],
        [3.9044e+00, 1.3683e-04],
        [3.6579e+00, 5.0445e-05],
        [3.4269e+00, 1.8598e-05],
        [3.2106e+00, 6.8565e-06],
        [3.0079e+00, 2.5278e-06],
        [2.8180e+00, 9.3194e-07],
        [2.6401e+00, 3.4358e-07],
        [2.4734e+00, 1.2667e-07],
        [2.3173e+00, 4.6700e-08],
        [2.1710e+00, 1.7217e-08],
        [2.0339e+00, 6.3476e-09],
        [1.9055e+00, 2.3402e-09],
        [1.7852e+00, 8.6277e-10],
        [1.6725e+00, 3.1808e-10],
        [1.5669e+00, 1.1727e-10],
        [1.4680e+00, 4.3234e-11],
        [1.3753e+00, 1.5939e-11],
        [1.2885e+00, 5.8764e-12],
        [1.207